# MemoryArena Kaggle EDA

This notebook performs exploratory data analysis and visualization for the MemoryArena benchmark. It is designed for a fresh Kaggle notebook: no Kaggle Input dataset is required. The notebook downloads the public Hugging Face dataset `ZexueHe/memoryarena`, clones the GitHub project, normalizes the five benchmark configs, and writes figures, tables, a Markdown report, and a zip archive under `/kaggle/working/memoryarena_eda_outputs`.

Kaggle note: turn **Internet** on in Notebook settings before running all cells.

## 1. Setup Environment

The setup cell installs missing packages, clones the project repository when running on Kaggle, prints key versions, and creates output directories.

In [ ]:
import importlib
import os
import platform
import subprocess
import sys
from pathlib import Path

PACKAGE_SPECS = [
    ("datasets", "datasets", False),
    ("pandas", "pandas", False),
    ("numpy", "numpy", False),
    ("matplotlib", "matplotlib", False),
    ("plotly", "plotly", False),
    ("networkx", "networkx", False),
    ("tqdm", "tqdm", False),
    ("sklearn", "scikit-learn", False),
    ("wordcloud", "wordcloud", True),
    ("tiktoken", "tiktoken", True),
]


def ensure_package(import_name, pip_name=None, optional=False):
    try:
        return importlib.import_module(import_name)
    except Exception:
        pip_name = pip_name or import_name
        print(f"Installing {pip_name} ...")
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])
            return importlib.import_module(import_name)
        except Exception as exc:
            if optional:
                print(f"Optional package {pip_name} unavailable: {exc}")
                return None
            raise


for import_name, pip_name, optional in PACKAGE_SPECS:
    ensure_package(import_name, pip_name, optional)

In [ ]:
REPO_URL = "https://github.com/toanthangO20/MemoryArena-Experiment.git"
KAGGLE_WORKING = Path("/kaggle/working")


def find_repo_root(start: Path) -> Path | None:
    for candidate in [start, *start.parents]:
        if (candidate / "README.md").exists() and (candidate / "agent").exists() and (candidate / "env").exists():
            return candidate
    return None


if KAGGLE_WORKING.exists():
    REPO_DIR = KAGGLE_WORKING / "MemoryArena-Experiment"
    if not REPO_DIR.exists():
        subprocess.check_call(["git", "clone", REPO_URL, str(REPO_DIR)])
else:
    REPO_DIR = find_repo_root(Path.cwd()) or Path.cwd()

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR / "src"))

from memoryarena_eda_utils import ensure_output_dirs

OUTPUT_DIR = KAGGLE_WORKING / "memoryarena_eda_outputs" if KAGGLE_WORKING.exists() else REPO_DIR / "memoryarena_eda_outputs"
OUTPUT_DIR, FIGURES_DIR, TABLES_DIR = ensure_output_dirs(OUTPUT_DIR)

print("Python:", sys.version)
print("Platform:", platform.platform())
print("Repo dir:", REPO_DIR)
print("Output dir:", OUTPUT_DIR)

for module_name in ["datasets", "pandas", "numpy", "matplotlib", "plotly", "networkx", "sklearn"]:
    module = importlib.import_module(module_name)
    print(f"{module_name}: {getattr(module, '__version__', 'unknown')}")

## 2. Load MemoryArena Dataset

The dataset is loaded directly from Hugging Face with `datasets.load_dataset("ZexueHe/memoryarena", config_name)`. The notebook uses the `test` split for each config and computes all counts from the downloaded data.

In [ ]:
import json
import math
import random
import re
import shutil
import warnings
from collections import Counter

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import plotly.express as px
from tqdm.auto import tqdm

from memoryarena_eda_utils import (
    CONSTRAINT_KEYWORDS,
    DEFAULT_CONFIGS,
    FORMAL_SUBSETS,
    SLOT_KEYWORDS,
    TRAVEL_KEYWORDS,
    build_formal_symbol_df,
    build_progressive_search_df,
    build_subtask_df,
    build_task_df,
    build_travel_df,
    compute_attribute_cooccurrence,
    compute_difficulty_score,
    estimate_tokens,
    extract_memory_units,
    extract_shopping_answers,
    extract_travel_edges,
    field_availability_by_subset,
    is_missing_value,
    load_memoryarena_subset,
    make_subset_summary,
    plot_bar_counts,
    plot_boxplot_by_group,
    plot_hist_by_group,
    plot_line,
    save_current_figure,
    write_eda_report,
)

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 160)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
CONFIGS = list(DEFAULT_CONFIGS)
print(CONFIGS)

In [ ]:
raw_records = {}
for config in tqdm(CONFIGS, desc="Loading MemoryArena configs"):
    records = load_memoryarena_subset(config)
    raw_records[config] = records
    print(f"{config}: {len(records):,} records")

actual_counts = {config: len(records) for config, records in raw_records.items()}
actual_counts

In [ ]:
task_df = build_task_df(raw_records)
subtask_df = build_subtask_df(raw_records)
task_df, difficulty_components = compute_difficulty_score(task_df, subtask_df, raw_records)
subset_summary = make_subset_summary(task_df, subtask_df)
field_availability = field_availability_by_subset(raw_records)

print("task_df", task_df.shape)
print("subtask_df", subtask_df.shape)
display(task_df.head())
display(subtask_df.head())

## 3. Data Quality and Schema Checks

This section checks record counts, field availability, field types, invalid schemas, and notable outliers. Schema validity here means `len(questions) == len(answers)`.

In [ ]:
display(subset_summary)

type_summary = (
    task_df.groupby(["subset", "questions_type", "answers_type", "backgrounds_type"])
    .size()
    .reset_index(name="count")
    .sort_values(["subset", "count"], ascending=[True, False])
)

missingness_rows = []
for subset, records in raw_records.items():
    for field in ["questions", "answers", "backgrounds", "base_person", "paper_name", "category"]:
        missing = sum(1 for record in records if is_missing_value(record.get(field)))
        missingness_rows.append({"subset": subset, "field": field, "missing_count": missing, "record_count": len(records)})
missingness_df = pd.DataFrame(missingness_rows)
missingness_df["missing_rate"] = missingness_df["missing_count"] / missingness_df["record_count"].replace(0, np.nan)

invalid_schema_df = task_df.loc[~task_df["schema_valid"], ["subset", "id", "num_questions", "num_answers"]]

print("Field type summary")
display(type_summary)
print("Missingness")
display(missingness_df)
print("Invalid schema rows")
display(invalid_schema_df.head(20))

In [ ]:
outlier_tables = {
    "many_subtasks": task_df.sort_values("num_questions", ascending=False).head(10),
    "long_questions": subtask_df.sort_values("question_tokens", ascending=False).head(10),
    "long_answers": subtask_df.sort_values("answer_tokens", ascending=False).head(10),
    "long_backgrounds": subtask_df.sort_values("background_tokens", ascending=False).head(10),
    "schema_invalid": invalid_schema_df.head(20),
}

for name, frame in outlier_tables.items():
    print("\n", name)
    display(frame)

In [ ]:
record_counts = task_df.groupby("subset").size().reset_index(name="record_count")
plot_bar_counts(record_counts, "subset", "record_count", "Record count by subset", FIGURES_DIR / "01_record_count_by_subset.png")

availability_pivot = field_availability.pivot(index="subset", columns="field", values="availability_rate").fillna(0)
fig, ax = plt.subplots(figsize=(9, 4.8))
im = ax.imshow(availability_pivot.values, aspect="auto", vmin=0, vmax=1, cmap="viridis")
ax.set_xticks(range(len(availability_pivot.columns)))
ax.set_xticklabels(availability_pivot.columns, rotation=30, ha="right")
ax.set_yticks(range(len(availability_pivot.index)))
ax.set_yticklabels(availability_pivot.index)
ax.set_title("Field availability by subset")
fig.colorbar(im, ax=ax, label="availability rate")
save_current_figure(FIGURES_DIR / "02_field_availability_heatmap.png")

invalid_counts = task_df.assign(invalid_schema=~task_df["schema_valid"]).groupby("subset")["invalid_schema"].sum().reset_index()
plot_bar_counts(invalid_counts, "subset", "invalid_schema", "Invalid schema count by subset", FIGURES_DIR / "03_invalid_schema_count.png")

## 4. General EDA

The general EDA compares multi-session length, text/token lengths, cumulative context pressure, answer structure, and task-level size across all five subsets.

In [ ]:
plot_hist_by_group(task_df, "num_questions", "subset", "Number of subtasks per task", FIGURES_DIR / "04_hist_num_subtasks_by_subset.png", bins=20)
plot_boxplot_by_group(task_df, "num_questions", "subset", "Subtasks per task by subset", FIGURES_DIR / "05_box_num_subtasks_by_subset.png")

plot_hist_by_group(subtask_df, "question_tokens", "subset", "Question token length distribution", FIGURES_DIR / "06_hist_question_tokens.png", bins=40)
plot_boxplot_by_group(subtask_df, "question_tokens", "subset", "Question token length by subset", FIGURES_DIR / "07_box_question_tokens.png")

plot_hist_by_group(subtask_df, "answer_tokens", "subset", "Answer token length distribution", FIGURES_DIR / "08_hist_answer_tokens.png", bins=40)
plot_boxplot_by_group(subtask_df, "answer_tokens", "subset", "Answer token length by subset", FIGURES_DIR / "09_box_answer_tokens.png")

plot_hist_by_group(subtask_df, "background_tokens", "subset", "Background token length distribution", FIGURES_DIR / "10_hist_background_tokens.png", bins=40)
plot_boxplot_by_group(subtask_df, "background_tokens", "subset", "Background token length by subset", FIGURES_DIR / "11_box_background_tokens.png")

In [ ]:
fig = px.scatter(
    task_df,
    x="num_questions",
    y="total_tokens_est",
    color="subset",
    hover_data=["id", "category", "paper_name", "proxy_difficulty_score"],
    title="Number of subtasks vs total estimated tokens",
)
fig.write_html(FIGURES_DIR / "12_scatter_num_questions_total_tokens.html")
fig.show()

context_curve = (
    subtask_df.groupby(["subset", "subtask_idx"])["cumulative_context_tokens"]
    .mean()
    .reset_index()
)
plot_line(
    context_curve,
    "subtask_idx",
    "cumulative_context_tokens",
    "subset",
    "Average cumulative context tokens by subtask index",
    FIGURES_DIR / "13_avg_cumulative_context_tokens.png",
)

In [ ]:
heatmap_source = subtask_df.copy()
heatmap_source["task_key"] = heatmap_source["subset"] + ":" + heatmap_source["task_id"].astype(str)
top_task_keys = heatmap_source.groupby("task_key")["question_tokens"].sum().sort_values(ascending=False).head(60).index
heatmap_sample = heatmap_source[heatmap_source["task_key"].isin(top_task_keys)]
heatmap_matrix = heatmap_sample.pivot_table(index="task_key", columns="subtask_idx", values="question_tokens", aggfunc="mean", fill_value=0)
fig, ax = plt.subplots(figsize=(12, max(6, 0.16 * len(heatmap_matrix))))
im = ax.imshow(heatmap_matrix.values, aspect="auto", cmap="magma")
ax.set_title("Question token heatmap for top token-heavy tasks")
ax.set_xlabel("subtask index")
ax.set_ylabel("task")
ax.set_xticks(range(len(heatmap_matrix.columns)))
ax.set_xticklabels(heatmap_matrix.columns)
ax.set_yticks(range(len(heatmap_matrix.index)))
ax.set_yticklabels(heatmap_matrix.index, fontsize=6)
fig.colorbar(im, ax=ax, label="question tokens")
save_current_figure(FIGURES_DIR / "14_task_subtask_question_token_heatmap.png")

answer_structure = task_df.groupby(["subset", "answer_structure_type"]).size().reset_index(name="count")
fig = px.bar(answer_structure, x="answer_structure_type", y="count", color="subset", barmode="group", title="Answer structure type by subset")
fig.write_html(FIGURES_DIR / "15_answer_structure_type_by_subset.html")
fig.show()

## 5. Subset EDA: `bundled_shopping`

Shopping tasks use structured answers with target ASINs and attributes. This section extracts product identifiers, attribute counts, top attributes, repeated ASINs, and attribute co-occurrence.

In [ ]:
shopping_answers_df = extract_shopping_answers(raw_records)
print(shopping_answers_df.shape)
display(shopping_answers_df.head())

category_counts = task_df[task_df["subset"] == "bundled_shopping"].groupby("category").size().reset_index(name="count")
plot_bar_counts(category_counts, "category", "count", "Shopping category distribution", FIGURES_DIR / "16_shopping_category_distribution.png", rotation=60)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(shopping_answers_df["num_attributes"], bins=range(0, int(shopping_answers_df["num_attributes"].max()) + 3), color="#4c78a8", alpha=0.8)
ax.set_title("Number of attributes per shopping answer")
ax.set_xlabel("attribute count")
ax.set_ylabel("answer count")
save_current_figure(FIGURES_DIR / "17_shopping_num_attributes_hist.png")

In [ ]:
attribute_counts = Counter()
for attrs in shopping_answers_df["attributes"]:
    attribute_counts.update(str(attr).lower() for attr in attrs)
attribute_top = pd.DataFrame(attribute_counts.most_common(30), columns=["attribute", "count"])
plot_bar_counts(attribute_top, "attribute", "count", "Top 30 shopping attributes", FIGURES_DIR / "18_shopping_top_attributes.png", rotation=70)

asin_counts = shopping_answers_df["target_asin"].value_counts().head(30).reset_index()
asin_counts.columns = ["target_asin", "count"]
plot_bar_counts(asin_counts, "target_asin", "count", "Top repeated ASINs", FIGURES_DIR / "19_shopping_top_asins.png", rotation=70)

shopping_task_stats = shopping_answers_df.groupby("task_id").agg(
    num_products=("subtask_idx", "count"),
    unique_asins=("target_asin", "nunique"),
    total_attributes=("num_attributes", "sum"),
).reset_index()
display(shopping_task_stats.describe())

In [ ]:
cooccurrence_df = compute_attribute_cooccurrence(shopping_answers_df, top_k=30)
if not cooccurrence_df.empty:
    top_edges = cooccurrence_df.sort_values("weight", ascending=False).head(80)
    G = nx.Graph()
    for _, row in top_edges.iterrows():
        G.add_edge(row["source"], row["target"], weight=float(row["weight"]))
    fig, ax = plt.subplots(figsize=(12, 9))
    pos = nx.spring_layout(G, seed=SEED, k=0.8)
    weights = [G[u][v]["weight"] for u, v in G.edges()]
    nx.draw_networkx_nodes(G, pos, node_size=500, node_color="#72b7b2", ax=ax)
    nx.draw_networkx_edges(G, pos, width=[0.4 + 0.25 * w for w in weights], alpha=0.35, ax=ax)
    nx.draw_networkx_labels(G, pos, font_size=8, ax=ax)
    ax.set_title("Shopping attribute co-occurrence network (top attributes)")
    ax.axis("off")
    save_current_figure(FIGURES_DIR / "20_shopping_attribute_cooccurrence_network.png")
else:
    print("No attribute co-occurrence edges available.")

## 6. Subset EDA: `progressive_search`

Progressive search tasks build final questions from accumulated intermediate constraints. The analysis tracks subquery count, unique answer count, repeated intermediate answers, final question patterns, and cumulative constraint keyword pressure.

In [ ]:
progressive_search_df = build_progressive_search_df(raw_records)
print(progressive_search_df.shape)
display(progressive_search_df.head())

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(progressive_search_df["num_subqueries"], bins=range(0, int(progressive_search_df["num_subqueries"].max()) + 2), color="#f58518", alpha=0.85)
ax.set_title("Progressive search subqueries per task")
ax.set_xlabel("number of subqueries")
ax.set_ylabel("task count")
save_current_figure(FIGURES_DIR / "21_progressive_num_subqueries_hist.png")

unique_answer_counts = progressive_search_df["unique_answer_count"].value_counts().sort_index().reset_index()
unique_answer_counts.columns = ["unique_answer_count", "task_count"]
plot_bar_counts(unique_answer_counts, "unique_answer_count", "task_count", "Unique answers per progressive task", FIGURES_DIR / "22_progressive_unique_answers.png", rotation=0)

consistency = progressive_search_df["all_intermediate_answers_same"].value_counts().reset_index()
consistency.columns = ["all_intermediate_answers_same", "task_count"]
plot_bar_counts(consistency, "all_intermediate_answers_same", "task_count", "Intermediate answer consistency", FIGURES_DIR / "23_progressive_answer_consistency.png", rotation=0)

In [ ]:
progressive_subtasks = subtask_df[subtask_df["subset"] == "progressive_search"].copy()
progressive_subtasks["cumulative_constraint_keywords"] = progressive_subtasks.groupby("task_id")["question_keyword_constraint_count"].cumsum()
constraint_curve = progressive_subtasks.groupby("subtask_idx")["cumulative_constraint_keywords"].mean().reset_index()
constraint_curve["subset"] = "progressive_search"
plot_line(constraint_curve, "subtask_idx", "cumulative_constraint_keywords", "subset", "Average cumulative constraint keywords", FIGURES_DIR / "24_progressive_cumulative_constraints.png")

last_idx = progressive_subtasks.groupby("task_id")["subtask_idx"].transform("max")
progressive_subtasks["question_position"] = np.where(progressive_subtasks["subtask_idx"] == last_idx, "final", "intermediate")
plot_boxplot_by_group(progressive_subtasks, "question_tokens", "question_position", "Final vs intermediate question token length", FIGURES_DIR / "25_progressive_final_vs_intermediate_tokens.png", rotation=0)

pattern_counts = progressive_search_df["final_question_patterns"].replace("", "none").str.get_dummies(sep="|").sum().sort_values(ascending=False).reset_index()
pattern_counts.columns = ["pattern", "task_count"]
plot_bar_counts(pattern_counts, "pattern", "task_count", "Final question pattern counts", FIGURES_DIR / "26_progressive_final_question_patterns.png", rotation=45)

## 7. Subset EDA: `group_travel_planner`

Travel planning tasks involve a base traveler, additional travelers, shared plans, constraints, and references across people. This section extracts traveler counts, trip days, slot-specific constraints, and dependency edges.

In [ ]:
travel_df = build_travel_df(raw_records)
travel_edges_df = extract_travel_edges(raw_records)
print(travel_df.shape, travel_edges_df.shape)
display(travel_df.head())
display(travel_edges_df.head())

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(travel_df["num_travelers"], bins=range(0, int(travel_df["num_travelers"].max()) + 2), color="#54a24b", alpha=0.85)
ax.set_title("Travelers per task")
ax.set_xlabel("travelers")
ax.set_ylabel("task count")
save_current_figure(FIGURES_DIR / "27_travel_travelers_per_task.png")

trip_days = travel_df["num_trip_days"].value_counts().sort_index().reset_index()
trip_days.columns = ["num_trip_days", "task_count"]
plot_bar_counts(trip_days, "num_trip_days", "task_count", "Trip days distribution", FIGURES_DIR / "28_travel_trip_days_distribution.png", rotation=0)

In [ ]:
travel_keyword_counts = []
for keyword in TRAVEL_KEYWORDS:
    col = f"kw_{keyword.replace(' ', '_')}"
    if col in travel_df:
        travel_keyword_counts.append({"keyword": keyword, "count": int(travel_df[col].sum())})
travel_keyword_counts = pd.DataFrame(travel_keyword_counts).sort_values("count", ascending=False)
plot_bar_counts(travel_keyword_counts, "keyword", "count", "Travel constraint keyword frequency", FIGURES_DIR / "29_travel_constraint_keywords.png", rotation=55)

slot_counts = []
for slot in SLOT_KEYWORDS:
    col = f"slot_{slot}"
    slot_counts.append({"slot": slot, "count": int(travel_df[col].sum())})
slot_counts = pd.DataFrame(slot_counts).sort_values("count", ascending=False)
plot_bar_counts(slot_counts, "slot", "count", "Travel slot keyword frequency", FIGURES_DIR / "30_travel_slot_keywords.png", rotation=30)

In [ ]:
if not travel_edges_df.empty:
    aggregate_edges = travel_edges_df.groupby(["source_person", "target_person"]).size().reset_index(name="weight").sort_values("weight", ascending=False).head(50)
    G = nx.DiGraph()
    for _, row in aggregate_edges.iterrows():
        G.add_edge(row["source_person"], row["target_person"], weight=int(row["weight"]))
    fig, ax = plt.subplots(figsize=(12, 9))
    pos = nx.spring_layout(G, seed=SEED, k=1.1)
    nx.draw_networkx_nodes(G, pos, node_size=520, node_color="#eeca3b", ax=ax)
    nx.draw_networkx_edges(G, pos, arrows=True, alpha=0.45, ax=ax)
    nx.draw_networkx_labels(G, pos, font_size=8, ax=ax)
    edge_labels = {(u, v): d["weight"] for u, v, d in G.edges(data=True)}
    nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=7, ax=ax)
    ax.set_title("Aggregate traveler dependency graph (top edges)")
    ax.axis("off")
    save_current_figure(FIGURES_DIR / "31_travel_dependency_graph.png")
else:
    print("No travel dependency edges detected.")

## 8. Subset EDA: Formal Reasoning

The formal reasoning configs contain paper names, questions, answers, and backgrounds. This section compares math and physics tasks, background length, notation reuse, and formula-heavy answer types.

In [ ]:
formal_task_df = task_df[task_df["subset"].isin(FORMAL_SUBSETS)].copy()
formal_subtask_df = subtask_df[subtask_df["subset"].isin(FORMAL_SUBSETS)].copy()
formal_symbol_df = build_formal_symbol_df(raw_records)
print(formal_task_df.shape, formal_subtask_df.shape, formal_symbol_df.shape)

paper_counts = formal_task_df.groupby(["subset", "paper_name"]).size().reset_index(name="task_count")
fig = px.bar(paper_counts.sort_values("task_count", ascending=False).head(40), x="paper_name", y="task_count", color="subset", title="Formal reasoning tasks by paper_name")
fig.write_html(FIGURES_DIR / "32_formal_tasks_by_paper_name.html")
fig.show()

plot_boxplot_by_group(formal_subtask_df, "background_tokens", "subset", "Formal background tokens by subset", FIGURES_DIR / "33_formal_background_tokens_box.png")

In [ ]:
formal_token_curve = formal_subtask_df.groupby(["subset", "subtask_idx"])[["question_tokens", "background_tokens", "answer_tokens"]].mean().reset_index()
fig, ax = plt.subplots(figsize=(10, 5.5))
for subset, frame in formal_token_curve.groupby("subset"):
    for value_col, style in [("question_tokens", "-"), ("background_tokens", "--"), ("answer_tokens", ":")]:
        ax.plot(frame["subtask_idx"], frame[value_col], style, marker="o", label=f"{subset} {value_col}")
ax.set_title("Formal average tokens by subtask index")
ax.set_xlabel("subtask index")
ax.set_ylabel("average tokens")
ax.legend(fontsize=7)
save_current_figure(FIGURES_DIR / "34_formal_tokens_by_subtask_index.png")

if not formal_symbol_df.empty:
    top_symbols = formal_symbol_df.groupby("symbol")["count"].sum().sort_values(ascending=False).head(35).reset_index()
    plot_bar_counts(top_symbols, "symbol", "count", "Top LaTeX/math symbols and terms", FIGURES_DIR / "35_formal_top_latex_symbols.png", rotation=70)

In [ ]:
if not formal_symbol_df.empty:
    selected_task = formal_symbol_df.groupby(["subset", "task_id"])["count"].sum().sort_values(ascending=False).index[0]
    selected_symbols = formal_symbol_df[(formal_symbol_df["subset"] == selected_task[0]) & (formal_symbol_df["task_id"] == selected_task[1])]
    top_task_symbols = selected_symbols.groupby("symbol")["count"].sum().sort_values(ascending=False).head(30).index
    reuse_matrix = selected_symbols[selected_symbols["symbol"].isin(top_task_symbols)].pivot_table(index="symbol", columns="subtask_idx", values="count", aggfunc="sum", fill_value=0)
    fig, ax = plt.subplots(figsize=(10, max(5, 0.22 * len(reuse_matrix))))
    im = ax.imshow(reuse_matrix.values, aspect="auto", cmap="cividis")
    ax.set_title(f"Notation reuse heatmap: {selected_task[0]} task {selected_task[1]}")
    ax.set_xlabel("subtask index")
    ax.set_ylabel("symbol")
    ax.set_xticks(range(len(reuse_matrix.columns)))
    ax.set_xticklabels(reuse_matrix.columns)
    ax.set_yticks(range(len(reuse_matrix.index)))
    ax.set_yticklabels(reuse_matrix.index, fontsize=7)
    fig.colorbar(im, ax=ax, label="symbol count")
    save_current_figure(FIGURES_DIR / "36_formal_notation_reuse_heatmap.png")

formal_compare = formal_task_df.groupby("subset")[["num_questions", "total_tokens_est", "total_background_tokens", "total_answer_tokens"]].mean().reset_index()
formal_compare_long = formal_compare.melt(id_vars="subset", var_name="feature", value_name="mean_value")
fig = px.bar(formal_compare_long, x="feature", y="mean_value", color="subset", barmode="group", title="Math vs physics formal reasoning comparison")
fig.write_html(FIGURES_DIR / "37_formal_math_vs_phys_comparison.html")
fig.show()

## 9. Proxy Difficulty Score

The proxy difficulty score is a tunable heuristic, not an official MemoryArena metric. It combines normalized task features: number of questions, token lengths, answer structure complexity, constraint keywords, entity or attribute count, and dependency references.

In [ ]:
difficulty_summary = task_df.groupby("subset").agg(
    avg_proxy_difficulty_score=("proxy_difficulty_score", "mean"),
    median_proxy_difficulty_score=("proxy_difficulty_score", "median"),
    avg_total_tokens=("total_tokens_est", "mean"),
    avg_num_questions=("num_questions", "mean"),
).reset_index().sort_values("avg_proxy_difficulty_score", ascending=False)
display(difficulty_summary)

plot_boxplot_by_group(task_df, "proxy_difficulty_score", "subset", "Proxy difficulty score by subset", FIGURES_DIR / "38_difficulty_boxplot_by_subset.png")
plot_bar_counts(difficulty_summary, "subset", "avg_proxy_difficulty_score", "Average proxy difficulty by subset", FIGURES_DIR / "39_difficulty_average_by_subset.png")

fig = px.scatter(
    task_df,
    x="total_tokens_est",
    y="proxy_difficulty_score",
    color="subset",
    hover_data=["id", "num_questions", "constraint_keyword_count", "entity_or_attribute_count"],
    title="Total estimated tokens vs proxy difficulty score",
)
fig.write_html(FIGURES_DIR / "40_difficulty_vs_total_tokens.html")
fig.show()

In [ ]:
component_cols = [col for col in difficulty_components.columns if col.endswith("_norm")]
component_by_subset = difficulty_components.merge(task_df[["subset", "id"]], on=["subset", "id"], how="left")
component_summary = component_by_subset.groupby("subset")[component_cols].mean().reset_index()
component_long = component_summary.melt(id_vars="subset", var_name="component", value_name="normalized_mean")
component_long["component"] = component_long["component"].str.replace("_norm", "", regex=False)
fig = px.bar(component_long, x="component", y="normalized_mean", color="subset", barmode="group", title="Difficulty component comparison by subset")
fig.write_html(FIGURES_DIR / "41_difficulty_components_grouped_bar.html")
fig.show()

## 10. Memory-Unit Analysis

Memory units are heuristic extractions of entities, attributes, constraints, relations, intermediate results, and background definitions. They are designed to reason about what an agent must remember, not to replace model-based evaluation.

In [ ]:
memory_units_df = extract_memory_units(raw_records)
print(memory_units_df.shape)
display(memory_units_df.head())

memory_type_counts = memory_units_df.groupby(["subset", "memory_type"]).size().reset_index(name="count")
fig = px.bar(memory_type_counts, x="subset", y="count", color="memory_type", title="Memory unit type distribution by subset")
fig.write_html(FIGURES_DIR / "42_memory_unit_type_distribution.html")
fig.show()

memory_pivot = memory_type_counts.pivot(index="subset", columns="memory_type", values="count").fillna(0)
fig, ax = plt.subplots(figsize=(10, 5.5))
bottom = np.zeros(len(memory_pivot))
for column in memory_pivot.columns:
    ax.bar(memory_pivot.index, memory_pivot[column], bottom=bottom, label=column)
    bottom += memory_pivot[column].values
ax.set_title("Stacked memory unit types by subset")
ax.set_xlabel("subset")
ax.set_ylabel("memory units")
ax.tick_params(axis="x", rotation=30)
ax.legend(fontsize=8)
save_current_figure(FIGURES_DIR / "43_memory_unit_stacked_bar.png")

In [ ]:
timeline_df = memory_units_df[memory_units_df["subtask_idx"] >= 0].groupby(["subset", "subtask_idx"]).size().reset_index(name="memory_unit_count")
plot_line(timeline_df, "subtask_idx", "memory_unit_count", "subset", "Memory unit count by subtask index", FIGURES_DIR / "44_memory_units_timeline.png")

## 11. Save Outputs

Tables, figures, and the Markdown report are written under `/kaggle/working/memoryarena_eda_outputs`. The final cell also creates `/kaggle/working/memoryarena_eda_outputs.zip`.

In [ ]:
# Required CSV outputs
task_df.to_csv(TABLES_DIR / "task_level_summary.csv", index=False)
subtask_df.to_csv(TABLES_DIR / "subtask_level_summary.csv", index=False)
shopping_answers_df.to_csv(TABLES_DIR / "shopping_answers.csv", index=False)
travel_edges_df.to_csv(TABLES_DIR / "travel_edges.csv", index=False)
memory_units_df.to_csv(TABLES_DIR / "memory_units.csv", index=False)
subset_summary.to_csv(TABLES_DIR / "subset_summary.csv", index=False)
difficulty_summary.to_csv(TABLES_DIR / "difficulty_summary.csv", index=False)

# Additional useful tables
field_availability.to_csv(TABLES_DIR / "field_availability.csv", index=False)
missingness_df.to_csv(TABLES_DIR / "missingness.csv", index=False)
type_summary.to_csv(TABLES_DIR / "schema_type_summary.csv", index=False)
progressive_search_df.to_csv(TABLES_DIR / "progressive_search_summary.csv", index=False)
travel_df.to_csv(TABLES_DIR / "travel_summary.csv", index=False)
formal_symbol_df.to_csv(TABLES_DIR / "formal_symbols.csv", index=False)
difficulty_components.to_csv(TABLES_DIR / "difficulty_components.csv", index=False)

write_eda_report(
    OUTPUT_DIR / "EDA_REPORT.md",
    CONFIGS,
    subset_summary,
    task_df,
    memory_units_df,
)

zip_path = shutil.make_archive(str(OUTPUT_DIR), "zip", root_dir=OUTPUT_DIR)
print("Saved tables to", TABLES_DIR)
print("Saved figures to", FIGURES_DIR)
print("Saved report to", OUTPUT_DIR / "EDA_REPORT.md")
print("Created zip", zip_path)

## 12. Next Steps

Review `EDA_REPORT.md`, inspect the generated figures, and use the CSV outputs for deeper model or memory-system analysis. The difficulty score is intentionally transparent and can be reweighted in `compute_difficulty_score`.